In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from utils import * 

# Load Data

In [ ]:
# df = load_all_parquet_files()
df = pd.read_csv("circuits_data414820.csv")

In [ ]:
dataFEATURES = df
print(dataFEATURES.shape)

# Find columns to drop (bad data columns)

In [ ]:
# dirty_counts = data.isna().sum()
dirty_counts = (
    dataFEATURES.replace(r"^\s*$", pd.NA, regex=True)
        .isna()
        .sum()
)

import matplotlib.pyplot as plt

ax = dirty_counts.plot(kind="barh", figsize=(10, 15))

for i, v in enumerate(dirty_counts):
    ax.text(v, i, str(v), va="center", ha="left")

plt.xlabel("Number of non-clean rows")
plt.ylabel("Feature")
plt.title("Non-clean rows per feature")
plt.tight_layout()
plt.show()



In [ ]:
DROP_COLS = [
    # ALMOST ALWAYS 1
    "connected_components",

    # WRONG FEATURES AND MANY MISSING VALUES
    "pagerank",
    "core_number",
    "sparsity",
    "memory_usage",
    "execution_time",
    "simulation_method",
    "automatic_transpiled_qubits",
    "transpiled_size",
    "transpiled_depth",
    "transpiled_qubits",
    "statevector_entropy",

    # REDUNDANT FEATURES CORRELATED WITH OTHERS
    "circuit_depth",
    "igdepth",
    "node_count",
    "circuit_qubits",

    # OLD RDBMS WRONG COLUMNS
    'iqs_ducksql_memory_mb',
    'iqs_ducksql_time_s',
    'iqs_np_mps_memory_mb',
    'iqs_np_mps_time_s',
    'iqs_np_one_shot_memory_mb',
    'iqs_np_one_shot_time_s',

    # OLD SHANNON ENTROPY
    # "statevector_saved_entropy",
]


dataFEATURES = dataFEATURES.drop(columns=DROP_COLS)

# Clean and Split into Tables and Save Schema

In [ ]:
import pandas as pd
from pathlib import Path

# -------------------------
# Start from features dataframe
# -------------------------
df = dataFEATURES.copy(deep=True)

out_dir = Path("clean_tables")
out_dir.mkdir(exist_ok=True)

KEY = "RowKey"

# -------------------------
# Qiskit simulation columns
# -------------------------
QISKIT_METHODS = [
    "statevector_saved", "statevector", "stabilizer", "density_matrix",
    "matrix_product_state", "extended_stabilizer", "unitary", "automatic"
]

QISKIT_SUFFIXES = [
    "transpiled_size",
    "transpiled_depth",
    "memory_usage",
    "execution_time",
    "gate_counts",
]

QISKIT_COLUMNS = [
    f"{m}_{s}" for m in QISKIT_METHODS for s in QISKIT_SUFFIXES
]

# -------------------------
# RDBMS simulation columns (programmatic)
# -------------------------
RDBMS_ENGINES = ["ducksql", "np_mps", "np_one_shot", "sqlite", "psql"]
RDBMS_METRICS = ["memory_mb", "time_s"]

RDBMS_COLUMNS = (
    ["infiniquantum_execution_time",
     "rdbms_execution_time",
     "infiniquantum_gate_counts",
     "infiniquantum_memory_usage",
     "infiniquantum_transpiled_depth",
     "infiniquantum_transpiled_size"]
    +
    [
        f"rdbms_{engine}_{metric}"
        for engine in RDBMS_ENGINES
        for metric in RDBMS_METRICS
    ]
)

# -------------------------
# SQL metrics
# -------------------------
SQL_COLUMNS = [
    'infinidata_quantum_sql_agg_func',
    'infinidata_quantum_sql_and',
    'infinidata_quantum_sql_cte',
    'infinidata_quantum_sql_eq_pred',
    'infinidata_quantum_sql_group_by',
    'infinidata_quantum_sql_num_join_occurrences',
    'infinidata_quantum_sql_num_joins',
    'infinidata_quantum_sql_num_unique_join_edges',
    'infinidata_quantum_sql_order_by',
    'infinidata_quantum_sql_select',
    'infinidata_quantum_sql_where'
]

# -------------------------
# Helper
# -------------------------
def save_clean(df_sub, name, require_any=False):
    cols = df_sub.columns.difference([KEY])
    if require_any:
        df_sub = df_sub.dropna(how="all", subset=cols)
    else:
        df_sub = df_sub.dropna()
    df_sub.to_csv(out_dir / name, index=False)

# -------------------------
# Simulation tables
# -------------------------
save_clean(
    df[[KEY] + QISKIT_COLUMNS],
    "qiskit_simulation.csv",
    require_any=True
)

save_clean(
    df[[KEY] + RDBMS_COLUMNS],
    "rdbms_simulation.csv",
    require_any=True
)

# -------------------------
# Merge old entropy into shannon entropy
# -------------------------
OLD_ENTROPY = "statevector_saved_entropy"
NEW_ENTROPY = "statevector_saved_shannon_entropy"

if OLD_ENTROPY in df.columns:
    # Fill shannon_entropy with old entropy where it's missing
    df[NEW_ENTROPY] = df[NEW_ENTROPY].combine_first(df[OLD_ENTROPY])


# -------------------------
# Metrics tables
# -------------------------
TABLE_SPECS = {
    "sql_metrics.csv": (
        SQL_COLUMNS,
        {
            'infinidata_quantum_sql_agg_func': 'num_agg_funcs',
            'infinidata_quantum_sql_and': 'num_and_clauses',
            'infinidata_quantum_sql_eq_pred': 'num_eq_predicates',
            'infinidata_quantum_sql_num_join_occurrences': 'num_join_occurrences',
            'infinidata_quantum_sql_num_unique_join_edges': 'num_unique_join_edges',
            'infinidata_quantum_sql_order_by': 'num_order_bys',
            'infinidata_quantum_sql_select': 'num_select_columns',
            'infinidata_quantum_sql_where': 'num_where_clauses',
            'infinidata_quantum_sql_num_joins': 'num_joins',
            'infinidata_quantum_sql_group_by': 'num_group_bys',
            'infinidata_quantum_sql_cte': 'num_subqueries_with'
        }
    ),
    "graph_metrics.csv": (
        [
            "edge_count", "max_degree", "min_cut_upper",
            "diameter", "radius", "average_degree",
            "average_clustering_coefficient",
            "average_shortest_path_length",
            "central_point_of_dominance",
            "std_dev_adjacency_matrix"
        ],
        None
    ),
    "static_metrics.csv": (
        [
            "num_qubits", "width", "depth", "circuit_size",
            "pauli_gate_count", "two_qubit_gate_count",
            "two_qubit_gate_percentage", "locality_ratio",
            "idling_score", "density_score", "gate_counts"
        ],
        None
    ),
    "dynamic_metrics.csv": (
        [
            # "statevector_saved_entropy",
            "statevector_saved_sparsity",
            "statevector_saved_shannon_entropy",
            "statevector_saved_von_neumann_entropy",
        ],
        None
    ),
}

for fname, (cols, rename_map) in TABLE_SPECS.items():
    sub = df[[KEY] + cols]
    if rename_map:
        sub = sub.rename(columns=rename_map)
    if "dynamic" in fname: save_clean(sub, fname, require_any=True)
    else: save_clean(sub, fname, require_any=False)
    # save_clean(sub, fname, require_any=False)

# -------------------------
# Metadata tables
# -------------------------
save_clean(
    df[[KEY, "PartitionKey", "blob_url"]],
    "azure_info.csv"
)

save_clean(
    df[[KEY, "name", "serialization_method", "qpy_serialization_success"]],
    "inferq_circuits.csv"
)

print(f"All clean tables saved to: {out_dir.resolve()}")


# Load Tables

In [ ]:
import pandas as pd
from pathlib import Path

# Directory where clean tables were saved
out_dir = Path("clean_tables")

# Tables to load (updated names)
tables = {
    "azure_info": "azure_info.csv",
    "inferq_circuits": "inferq_circuits.csv",
    "graph_metrics": "graph_metrics.csv",
    "static_metrics": "static_metrics.csv",
    "dynamic_metrics": "dynamic_metrics.csv",
    "sql_metrics": "sql_metrics.csv",
    "qiskit_simulation": "qiskit_simulation.csv",
    "rdbms_simulation": "rdbms_simulation.csv",
}

# Load and count rows
row_counts = []

for name, fname in tables.items():
    df = pd.read_csv(out_dir / fname)
    row_counts.append({
        "table": name,
        "rows": len(df)
    })

# Display results
counts_df = (
    pd.DataFrame(row_counts)
    .sort_values("table")
    .reset_index(drop=True)
)

counts_df


In [ ]:
# # Entropy SANITY CHECK:
# dynamic_df = pd.read_csv(out_dir / "dynamic_metrics.csv")
# # Let us find out how many times statevector_saved_shannon entropy is equal to statevector_saved_entropy
# num_equal = (np.isclose(dynamic_df["statevector_saved_shannon_entropy"], dynamic_df["statevector_saved_entropy"])).sum()
# num_total = len(dynamic_df)
# print(f"Number of equal entries: {num_equal} out of {num_total}")
# print(f"Proportion of equal entries: {num_equal / num_total:.4f}")

In [ ]:
import pandas as pd
from pathlib import Path

base = Path("clean_tables")

def print_schema(df, name):
    print(f"\nSchema for {name}")
    print("=" * (len(name) + 11))
    for col in df.columns:
        print(f"{col:<50} {df[col].dtype}")

for name, fname in tables.items():
    df = pd.read_csv(base / fname)
    print_schema(df, name)


In [ ]:
import pandas as pd
from pathlib import Path

# Load qiskit metrics
qiskit_df = pd.read_csv(Path("clean_tables") / "qiskit_simulation.csv")

key = "RowKey"
sim_cols = [c for c in qiskit_df.columns if c != key]

# Split columns by method
automatic_cols = [c for c in sim_cols if c.startswith("automatic_")]
non_automatic_cols = [c for c in sim_cols if not c.startswith("automatic_")]

# Boolean masks
at_least_one_sim = qiskit_df[sim_cols].notna().any(axis=1)
all_sim_worked = qiskit_df[sim_cols].notna().all(axis=1)

auto_worked = qiskit_df[automatic_cols].notna().any(axis=1)

only_auto_worked = (
    qiskit_df[automatic_cols].notna().any(axis=1)
    & qiskit_df[non_automatic_cols].isna().all(axis=1)
)

# Print results
print("At least 1 sim worked")
print(at_least_one_sim.sum())

print("\nAll sim worked:")
print(all_sim_worked.sum())

print("\nAuto worked")
print(auto_worked.sum())

print("\nOnly automatic worked:")
print(only_auto_worked.sum())


In [ ]:
import pandas as pd
from pathlib import Path

base = Path("clean_tables")
KEY = "RowKey"

# Load tables
dynamic_df = pd.read_csv(base / "dynamic_metrics.csv")
static_df  = pd.read_csv(base / "static_metrics.csv")
graph_df   = pd.read_csv(base / "graph_metrics.csv")
qiskit_df  = pd.read_csv(base / "qiskit_simulation.csv")

# Join cleaned feature tables
features_df = (
    dynamic_df
    .merge(static_df, on=KEY, how="inner")
    .merge(graph_df, on=KEY, how="inner")
)

# Join with Qiskit simulation
joined_df = features_df.merge(qiskit_df, on=KEY, how="inner")

print("Rows after join (Dynamic + Static + Graph + Qiskit):")
print(len(joined_df))

# Qiskit simulation columns
sim_cols = qiskit_df.columns.difference([KEY])
automatic_cols = [c for c in sim_cols if c.startswith("automatic_")]
non_automatic_cols = [c for c in sim_cols if not c.startswith("automatic_")]

# Masks (on joined set)
at_least_one_sim = joined_df[sim_cols].notna().any(axis=1)
auto_worked = joined_df[automatic_cols].notna().any(axis=1)
only_auto_worked = auto_worked & joined_df[non_automatic_cols].isna().all(axis=1)

print("\nAt least 1 simulation worked:")
print(at_least_one_sim.sum())

print("\nAutomatic worked:")
print(auto_worked.sum())

print("\nOnly automatic worked:")
print(only_auto_worked.sum())


In [ ]:
import pandas as pd
from pathlib import Path

base = Path("clean_tables")
KEY = "RowKey"

# -------------------------------------------------
# Load tables
# -------------------------------------------------
sql_df    = pd.read_csv(base / "sql_metrics.csv")
rdbms_df = pd.read_csv(base / "rdbms_simulation.csv")

# -------------------------------------------------
# Join SQL features with RDBMS simulations
# -------------------------------------------------
joined_df = sql_df.merge(rdbms_df, on=KEY, how="inner")

print("Rows after join (SQL + RDBMS):")
print(len(joined_df))

# -------------------------------------------------
# RDBMS simulation columns
# -------------------------------------------------
sim_cols = rdbms_df.columns.difference([KEY])

# group by backend
RDBMS_METHODS = [
    "psql",
    "ducksql",
    "sqlite",
    "infiniquantum"
]

method_cols = {
    m: [c for c in sim_cols if m in c]
    for m in RDBMS_METHODS
}

# -------------------------------------------------
# Masks
# -------------------------------------------------
# At least one backend worked
at_least_one_sim = joined_df[sim_cols].notna().any(axis=1)

# Per-backend success
method_worked = {
    m: joined_df[cols].notna().any(axis=1)
    for m, cols in method_cols.items()
}

# Exactly one backend worked
only_one_worked = (
    pd.concat(method_worked.values(), axis=1).sum(axis=1) == 1
)

# -------------------------------------------------
# Report
# -------------------------------------------------
print("\nAt least one RDBMS simulation worked:")
print(at_least_one_sim.sum())

print("\nPer-backend success counts:")
for m, mask in method_worked.items():
    print(f"{m:15s}: {mask.sum()}")

print("\nOnly one backend worked:")
print(only_one_worked.sum())


# Training Data Creation

# Centralized Training Data Creation
This section generates training datasets for different models (Selector, Estimator, RDBMS) and saves them as Parquet files in `analysis/training_data/`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import ast
from utils import clean_dataframe  # Ensure utils.py is in the python path or same directory

# Common Setup
base = Path("clean_tables")
output_dir = Path("training_data")
output_dir.mkdir(exist_ok=True)
PRIMARY_KEY = "RowKey"

## 1. Selector Training Data
Generates data for selecting the best simulation method.
*   Merges dynamic, static, graph features with Qiskit simulation results.
*   Calculates `best_method` target based on execution time.
*   Output: `selector_training_data.parquet`

In [ ]:
# 1. Load feature tables
dynamic_df = pd.read_csv(base / "dynamic_metrics.csv")
dynamic_df = clean_dataframe(dynamic_df, "statevector_saved_von_neumann_entropy")

static_df  = pd.read_csv(base / "static_metrics.csv")
graph_df   = pd.read_csv(base / "graph_metrics.csv")
qiskit_df  = pd.read_csv(base / "qiskit_simulation.csv")

# 2. Merge feature tables
feature_df = dynamic_df.merge(static_df, on=PRIMARY_KEY)\
                       .merge(graph_df, on=PRIMARY_KEY)

# 3. Select relevant Qiskit columns
keep_suffixes = ["memory_usage", "execution_time"]
qiskit_cols = [PRIMARY_KEY] + [c for c in qiskit_df.columns if any(c.endswith(s) for s in keep_suffixes)]
qiskit_df_subset = qiskit_df[qiskit_cols]

# 4. Merge features with simulation data
df_selector = feature_df.merge(qiskit_df_subset, on=PRIMARY_KEY)

# 5. Filter valid rows (where automatic simulation ran)
auto_cols = [c for c in df_selector.columns if c.startswith("automatic_")]
df_selector = df_selector[df_selector[auto_cols].notna().any(axis=1)].copy()

# 6. Define Methods and Target Calculation
METHODS_SELECTOR = [
    "density_matrix",
    "matrix_product_state",
    "extended_stabilizer",
    "statevector",
]

def get_best_method(row):
    costs = {}
    for m in METHODS_SELECTOR:
        t = row.get(f"{m}_execution_time")
        mem = row.get(f"{m}_memory_usage")
        if pd.isna(t) or pd.isna(mem):
            continue
        costs[m] = t
    return min(costs, key=costs.get) if costs else np.nan

df_selector["best_method"] = df_selector.apply(get_best_method, axis=1)
df_selector = df_selector.dropna(subset=["best_method"])

# 7. Save
output_path = output_dir / "selector_training_data.parquet"
df_selector.to_parquet(output_path)
# df_selector = df_selector.copy()

# # Convert Arrow-backed dtypes → numpy-backed
# df_selector = df_selector.convert_dtypes(dtype_backend="numpy_nullable")

# # Convert Period columns explicitly
# for col in df_selector.columns:
#     if str(df_selector[col].dtype).startswith("period"):
#         df_selector[col] = df_selector[col].astype(str)

# df_selector.to_parquet(output_path, engine="pyarrow")


print(f"Saved Selector training data to {output_path} ({len(df_selector)} rows)")

## 2. Estimator Training Data
Generates data for estimating quantum properties (sparsity, entropy).
*   Aggregates Von Neumann entropy list features.
*   Applies log transformations to saturated variables.
*   Creates sparsity bins for classification.
*   Output: `estimator_training_data.parquet`

In [ ]:
# 1. Load tables (Dynamic, Static, Graph)
dynamic_df = pd.read_csv(base / "dynamic_metrics.csv")
# Note: we don't necessarily need clean_dataframe here if we re-parse using ast.literal_eval, 
# but consistent cleaning is good.
static_df  = pd.read_csv(base / "static_metrics.csv")
graph_df   = pd.read_csv(base / "graph_metrics.csv")

df_estimator = dynamic_df.merge(static_df, on=PRIMARY_KEY)\
                         .merge(graph_df, on=PRIMARY_KEY)


# 5. Save
output_path = output_dir / "estimator_training_data.parquet"
df_estimator.to_parquet(output_path)
print(f"Saved Estimator training data to {output_path} ({len(df_estimator)} rows)")

## 3. RDBMS Training Data
Generates data for comparing RDBMS vs Qiskit simulation performance.
*   Merges SQL metrics with other features and simulation results (RDBMS & Qiskit).
*   Calculates best method across all simulators (SQL + Quantum).
*   Output: `rdbms_training_data.parquet`

In [ ]:
# 1. Load all tables
sql_df     = pd.read_csv(base / "sql_metrics.csv")
static_df  = pd.read_csv(base / "static_metrics.csv")
dynamic_df = pd.read_csv(base / "dynamic_metrics.csv")
dynamic_df = clean_dataframe(dynamic_df, "statevector_saved_von_neumann_entropy")
graph_df   = pd.read_csv(base / "graph_metrics.csv")
qiskit_df  = pd.read_csv(base / "qiskit_simulation.csv")
rdbms_df   = pd.read_csv(base / "rdbms_simulation.csv")

# 2. Merge Features
feature_df = sql_df.merge(static_df, on=PRIMARY_KEY, how="outer") \
                   .merge(dynamic_df, on=PRIMARY_KEY, how="outer") \
                   .merge(graph_df, on=PRIMARY_KEY, how="outer")
FEATURES = feature_df.select_dtypes(include=[np.number]).columns.tolist()
print(FEATURES)
# 3. Filter RDBMS columns
keep_suffixes = ["mb", "time_s"]
rdbms_cols = [PRIMARY_KEY] + [c for c in rdbms_df.columns if any(c.endswith(s) for s in keep_suffixes)]
rdbms_df_subset = rdbms_df[rdbms_cols]

# 4. Merge simulation results
df_rdbms = feature_df.merge(rdbms_df_subset, on=PRIMARY_KEY)
# Merge Qiskit for comparison (suffixes handled automatically or specified)
df_rdbms = df_rdbms.merge(qiskit_df, on=PRIMARY_KEY, suffixes=("", "_qiskit"))

# 5. Filtering rows with valid 'sqlite' simulation (assumed baseline)
# Checking for columns containing 'sqlite' in the original rdbms dataframe (excluding PK)
sim_cols_check = [c for c in df_rdbms.columns if c != PRIMARY_KEY and 'sqlite' in c]
# We need to check existence in merged df
sim_cols_check = [c for c in sim_cols_check if c in df_rdbms.columns]
if sim_cols_check:
    df_rdbms = df_rdbms.dropna(subset=sim_cols_check, how="any")

# 6. Drop Missing features
df_rdbms=df_rdbms.dropna(subset=FEATURES,how="any")

# 7. Compute Best Method (RDBMS vs Qiskit)
RDBMS_METHODS = ["sqlite", "ducksql", "psql"]
QISKIT_METHODS = ["density_matrix", "matrix_product_state", "extended_stabilizer", "statevector"]
ALL_METHODS = RDBMS_METHODS + QISKIT_METHODS

# Vectorized min calculation for Time and Memory
time_cols = {
    m: [c for c in df_rdbms.columns if m in c and c.endswith(("time_s", "execution_time"))] 
    for m in ALL_METHODS
}
mem_cols = {
    m: [c for c in df_rdbms.columns if m in c and c.endswith(("mb", "memory_usage"))] 
    for m in ALL_METHODS
}

# Helper to find min across columns
time_min_df = pd.DataFrame({
    m: df_rdbms[cols].min(axis=1) for m, cols in time_cols.items() if cols
})
mem_min_df = pd.DataFrame({
    m: df_rdbms[cols].min(axis=1) for m, cols in mem_cols.items() if cols
})

df_rdbms["best_time_method"] = time_min_df.idxmin(axis=1)
df_rdbms["best_mem_method"]  = mem_min_df.idxmin(axis=1)

# 7. Save
output_path = output_dir / "rdbms_training_data.parquet"
df_rdbms.to_parquet(output_path)
print(f"Saved RDBMS training data to {output_path} ({len(df_rdbms)} rows)")

In [ ]:
# 1. Load all tables
sql_df     = pd.read_csv(base / "sql_metrics.csv")
static_df  = pd.read_csv(base / "static_metrics.csv")
dynamic_df = pd.read_csv(base / "dynamic_metrics.csv")
# dynamic_df = clean_dataframe(dynamic_df, "statevector_saved_von_neumann_entropy") # Optional if we don't filter by features
graph_df   = pd.read_csv(base / "graph_metrics.csv")
qiskit_df  = pd.read_csv(base / "qiskit_simulation.csv")
rdbms_df   = pd.read_csv(base / "rdbms_simulation.csv")

# 2. Merge all dataframes (Outer Join to keep all rows initially)
df_all_methods = sql_df.merge(static_df, on=PRIMARY_KEY, how="outer") \
                       .merge(dynamic_df, on=PRIMARY_KEY, how="outer") \
                       .merge(graph_df, on=PRIMARY_KEY, how="outer") \
                       .merge(rdbms_df, on=PRIMARY_KEY, how="outer") \
                       .merge(qiskit_df, on=PRIMARY_KEY, how="outer", suffixes=("", "_qiskit"))

# 3. Define all simulation methods we care about
# Using the same list as in the previous cell to be consistent with "all simulation methods" used for RDBMS comparison
RDBMS_METHODS = ["sqlite", "ducksql"]
QISKIT_METHODS = ["density_matrix", "matrix_product_state", "extended_stabilizer", "statevector"]
ALL_METHODS = RDBMS_METHODS + QISKIT_METHODS

# 4. Identify Time and Memory columns for ALL methods
required_cols = []
for m in ALL_METHODS:
    # Find columns for this method (time and memory)
    # We look for columns containing the method name and the appropriate suffix
    # Note: RDBMS cols are like rdbms_{method}_{metric}, Qiskit cols are like {method}_{metric}
    # The suffixes in the merged df might handle collisions, but generally:
    # Qiskit: "execution_time", "memory_usage"
    # RDBMS: "time_s", "mb" (inside the column name usually)
    
    # Logic adapted from previous cell to find the exact column names present in df_all_methods
    m_cols = [c for c in df_all_methods.columns if m in c]
    
    # Filter for time/memory keywords
    m_time = [c for c in m_cols if c.endswith("time_s") or c.endswith("execution_time")]
    m_mem  = [c for c in m_cols if c.endswith("mb") or c.endswith("memory_usage")]
    
    required_cols.extend(m_time)
    required_cols.extend(m_mem)

# Remove duplicates if any
required_cols = list(set(required_cols))

print(f"Filtering based on {len(required_cols)} columns for {len(ALL_METHODS)} methods.")

# 5. Filter: Keep only rows where WE HAVE DATA for ALL methods (time AND memory)
# We do NOT filter based on FEATURES (dynamic/static/etc)
df_all_methods_filtered = df_all_methods.dropna(subset=required_cols, how="any")

# 6. Save
output_path = output_dir / "rdbms_no_psql_methods_training_data.parquet"
df_all_methods_filtered.to_parquet(output_path)
print(f"Saved RDBMS All Methods training data to {output_path} ({len(df_all_methods_filtered)} rows)")
